# Lipschitz Constant Measurement - Testing Paper's Assumption

**Goal**: Prove L_x,w can be >> 1, breaking the paper's CWF guarantee (CONTRIBUTIONS.md)

**Method**: 
1. Baseline: Measure L_x,w for standard watermarks (expect ≈1-5)
2. Adversarial: Use PGD to maximize ||φ(x+δ) - φ(x)||² subject to ||δ|| ≤ Δ
3. Compare: Show L_adv >> L_baseline, proving assumption violation

In [ ]:
!pip install -r requirements.txt

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import os
from PIL import Image
from tqdm import tqdm
from regen_pipe import ReSDPipeline
from watermarker import InvisibleWatermarker
from utils import eval_psnr_ssim_msssim
import matplotlib.pyplot as plt

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
image_dir = '/content/drive/MyDrive/lipschitz_test_imgs/'
output_dir = '/content/drive/MyDrive/lipschitz_results/'
os.makedirs(output_dir, exist_ok=True)

## Load Stable Diffusion VAE (φ function from paper's equation 2)

In [ ]:
# Load SD VAE encoder as φ (embedding function in paper's framework)
pipe = ReSDPipeline.from_pretrained(
    "Manojb/stable-diffusion-2-1-base",
    torch_dtype=torch.float32 if device == "cpu" else torch.float16
)
pipe.to(device)
vae = pipe.vae  # φ in equation (2): z_t = √α(t)·E(x_w) + N(0,(1-α(t))I_d)
print('Model loaded')

## Helper Functions for Lipschitz Measurement (CONTRIBUTIONS.md Step 3)

In [ ]:
def load_image_tensor(path, device):
    img = Image.open(path).convert('RGB').resize((512, 512))
    img = np.asarray(img) / 255.0
    img = (img - 0.5) * 2
    return torch.tensor(img, dtype=vae.dtype, device=device).permute(2, 0, 1).unsqueeze(0)

def get_latent(img_tensor, vae):
    """Compute φ(x) using VAE encoder"""
    with torch.no_grad():
        return vae.encode(img_tensor).latent_dist.mean

def compute_lipschitz(x, x_w, vae):
    """Calculate L_x,w = ||φ(x_w) - φ(x)|| / ||x_w - x|| (CONTRIBUTIONS.md)"""
    pixel_dist = torch.norm(x_w - x).item()  # ||x_w - x|| in pixel space
    latent_x = get_latent(x, vae)
    latent_xw = get_latent(x_w, vae)
    latent_dist = torch.norm(latent_xw - latent_x).item()  # ||φ(x_w) - φ(x)|| in latent space
    return latent_dist / pixel_dist if pixel_dist > 0 else 0, pixel_dist, latent_dist

## Step 1: Baseline - Standard Watermarks (CONTRIBUTIONS.md Step 2)

**Expected**: L_x,w ≈ 1-5 (embedding doesn't amplify watermark)

In [ ]:
# Standard watermarking methods for baseline comparison
wmarkers = {
    'DwtDct': InvisibleWatermarker('test', 'dwtDct'),
    'DwtDctSvd': InvisibleWatermarker('test', 'dwtDctSvd'),
    'RivaGAN': InvisibleWatermarker('test', 'rivaGan'),
}

In [ ]:
import glob
image_paths = sorted(glob.glob(os.path.join(image_dir, '*.png')))[:20]
print(f'Found {len(image_paths)} images')

In [ ]:
# Measure baseline L_x,w for standard watermarks
baseline_results = {}

for wm_name, wmarker in wmarkers.items():
    print(f'\nProcessing {wm_name}...')
    lipschitz_constants = []
    pixel_dists = []
    latent_dists = []
    psnr_vals, ssim_vals, msssim_vals = [], [], []
    
    wm_save_dir = os.path.join(output_dir, 'standard_watermarks', wm_name)
    os.makedirs(wm_save_dir, exist_ok=True)
    
    for img_path in tqdm(image_paths):
        wm_path = os.path.join(wm_save_dir, os.path.basename(img_path))
        wmarker.encode(img_path, wm_path)
        
        x = load_image_tensor(img_path, device)
        x_w = load_image_tensor(wm_path, device)
        
        L, pd, ld = compute_lipschitz(x, x_w, vae)
        lipschitz_constants.append(L)
        pixel_dists.append(pd)
        latent_dists.append(ld)
        psnr, ssim_val, msssim_val = eval_psnr_ssim_msssim(img_path, wm_path)
        psnr_vals.append(psnr)
        ssim_vals.append(ssim_val)
        msssim_vals.append(msssim_val)
    
    baseline_results[wm_name] = {
        'L_mean': np.mean(lipschitz_constants),
        'L_std': np.std(lipschitz_constants),
        'pixel_dist_mean': np.mean(pixel_dists),
        'latent_dist_mean': np.mean(latent_dists),
        'psnr_mean': np.mean(psnr_vals),
        'ssim_mean': np.mean(ssim_vals),
        'msssim_mean': np.mean(msssim_vals)
    }
    print(f'{wm_name}: L = {baseline_results[wm_name]["L_mean"]:.4f} ± {baseline_results[wm_name]["L_std"]:.4f}, '
      f'PSNR = {baseline_results[wm_name]["psnr_mean"]:.2f}, SSIM = {baseline_results[wm_name]["ssim_mean"]:.4f}, MSSSIM = {baseline_results[wm_name]["msssim_mean"]:.4f}')

## Step 2: Adversarial Watermark via PGD (CONTRIBUTIONS.md Step 3)

**Objective**: Maximize J = ||φ(x+δ) - φ(x)||² subject to ||δ|| ≤ ε

**Rationale**: Paper admits "if watermark is carefully designed...aligned with adversarial perturbation, L_x,w >> 1" (CONTEXT.md)

In [ ]:
def pgd_adversarial_watermark(x_wm, x_clean, vae, epsilon=8/255, alpha=2/255, steps=40):
    """PGD to amplify latent distance of existing watermark while preserving message"""
    delta = torch.zeros_like(x_wm, requires_grad=True)
    
    with torch.no_grad():
        latent_clean = vae.encode(x_clean).latent_dist.mean
    
    for _ in range(steps):
        if delta.grad is not None:
            delta.grad.zero_()
        
        x_adv = x_wm + delta
        latent_adv = vae.encode(x_adv).latent_dist.mean
        loss = -torch.norm(latent_adv - latent_clean)**2
        loss.backward()
        
        with torch.no_grad():
            delta.data = delta.data - alpha * delta.grad.sign()
            total_pert = x_adv - x_clean
            pert_norm = torch.norm(total_pert.reshape(total_pert.shape[0], -1), dim=1, keepdim=True)
            if pert_norm > epsilon:
                total_pert = total_pert * (epsilon / (pert_norm.view(-1, 1, 1, 1) + 1e-8))
                delta.data = (x_clean + total_pert - x_wm).clamp(-1, 1)
        
        delta.requires_grad = True
    
    return (x_wm + delta).detach()

## Step 3: Detection Function

**Verify watermark message is preserved using bit accuracy**

In [ ]:
from utils import bytearray_to_bits
from scipy.stats import binomtest

def detect_watermark(img_path, wmarker, threshold=23, k=32):
    """Detect watermark using wmarker.decode(). Returns (detected, bit_accuracy, p_value)"""
    try:
        wm_decoded = wmarker.decode(img_path)
        bits_decoded = bytearray_to_bits(wm_decoded)[:k]
        bits_original = bytearray_to_bits(wmarker.wm_text.encode('utf-8'))[:k]
        matches = sum(b1 == b2 for b1, b2 in zip(bits_decoded, bits_original))
        bit_acc = matches / k
        p_val = binomtest(matches, k, 0.5, alternative='greater').pvalue
        detected = matches >= threshold
        return detected, bit_acc, p_val
    except:
        return False, 0.0, 1.0

## Step 4: Generate and Measure Adversarial Watermarks

In [ ]:
base_wmarker = InvisibleWatermarker('test', 'dwtDct')
adv_save_dir = os.path.join(output_dir, 'adversarial_watermarks')
os.makedirs(adv_save_dir, exist_ok=True)

lipschitz_constants_adv = []
pixel_dists_adv = []
latent_dists_adv = []
psnr_vals_adv, ssim_vals_adv, msssim_vals_adv = [], [], []
detection_results = []

print('Generating adversarial watermarks...')
for img_path in tqdm(image_paths):
    # Step 1: Apply base watermark (encodes message)
    base_wm_path = os.path.join(adv_save_dir, 'base_' + os.path.basename(img_path))
    base_wmarker.encode(img_path, base_wm_path)
    
    # Step 2: Load tensors
    x_clean = load_image_tensor(img_path, device)
    x_wm = load_image_tensor(base_wm_path, device)
    
    # Step 3: Apply PGD to amplify latent distance
    x_adv = pgd_adversarial_watermark(x_wm, x_clean, vae)
    
    # Step 4: Save adversarial watermarked image
    adv_path = os.path.join(adv_save_dir, os.path.basename(img_path))
    x_adv_img = (x_adv.squeeze().permute(1, 2, 0).cpu().numpy() + 1) / 2 * 255
    x_adv_img = np.clip(x_adv_img, 0, 255).astype(np.uint8)
    cv2.imwrite(adv_path, cv2.cvtColor(x_adv_img, cv2.COLOR_RGB2BGR))
    
    # Step 5: Measure Lipschitz constant
    L, pd, ld = compute_lipschitz(x_clean, x_adv, vae)
    lipschitz_constants_adv.append(L)
    pixel_dists_adv.append(pd)
    latent_dists_adv.append(ld)
    
    # Step 6: Measure image quality
    psnr, ssim_val, msssim_val = eval_psnr_ssim_msssim(img_path, adv_path)
    psnr_vals_adv.append(psnr)
    ssim_vals_adv.append(ssim_val)
    msssim_vals_adv.append(msssim_val)
    
    # Step 7: Verify watermark detection
    detected, bit_acc, p_val = detect_watermark(adv_path, base_wmarker)
    detection_results.append({'detected': detected, 'bit_acc': bit_acc, 'p_val': p_val})

adversarial_results = {
    'L_mean': np.mean(lipschitz_constants_adv),
    'L_std': np.std(lipschitz_constants_adv),
    'pixel_dist_mean': np.mean(pixel_dists_adv),
    'latent_dist_mean': np.mean(latent_dists_adv),
    'psnr_mean': np.mean(psnr_vals_adv),
    'ssim_mean': np.mean(ssim_vals_adv),
    'msssim_mean': np.mean(msssim_vals_adv)
}
print(f'Adversarial: L = {adversarial_results["L_mean"]:.4f} ± {adversarial_results["L_std"]:.4f}, '
      f'PSNR = {adversarial_results["psnr_mean"]:.2f}, SSIM = {adversarial_results["ssim_mean"]:.4f}, MSSSIM = {adversarial_results["msssim_mean"]:.4f}')

print('\n=== WATERMARK DETECTION ===')
det_rate = np.mean([r['detected'] for r in detection_results])
avg_bit_acc = np.mean([r['bit_acc'] for r in detection_results])
print(f'Adversarial watermark detection rate: {det_rate*100:.1f}%')
print(f'Average bit accuracy: {avg_bit_acc:.3f}')
print(f'Message preserved: {det_rate > 0.9}')

In [ ]:
print('\n=== LIPSCHITZ CONSTANT COMPARISON ===')
print('\nBaseline (Standard Watermarks):')
for wm_name, results in baseline_results.items():
    print(f'  {wm_name:15s}: L = {results["L_mean"]:.4f} ± {results["L_std"]:.4f}')

print('\nAdversarial Watermark:')
print(f'  PGD Attack      : L = {adversarial_results["L_mean"]:.4f} ± {adversarial_results["L_std"]:.4f}')

print('\n=== VIOLATION ANALYSIS ===')
baseline_avg = np.mean([r['L_mean'] for r in baseline_results.values()])
ratio = adversarial_results['L_mean'] / baseline_avg
print(f'Adversarial L is {ratio:.2f}x larger than baseline average')
print(f'Lipschitz property violated: {ratio > 10}')

print('\n=== IMAGE QUALITY ASSESSMENT ===')
print('Watermarks maintain invisibility (high PSNR/SSIM):')
for wm_name, results in baseline_results.items():
    print(f'  {wm_name:15s}: PSNR={results["psnr_mean"]:.2f}dB, SSIM={results["ssim_mean"]:.4f}, MSSSIM={results["msssim_mean"]:.4f}')
print(f'  Adversarial    : PSNR={adversarial_results["psnr_mean"]:.2f}dB, SSIM={adversarial_results["ssim_mean"]:.4f}, MSSSIM={adversarial_results["msssim_mean"]:.4f}')
    
print('\n=== THEORETICAL IMPLICATIONS ===')
print(f'For CWF guarantee, attacker needs sigma/Delta proportional to L:')
print(f'  Baseline: sigma/Delta ~ {baseline_avg:.2f}')
print(f'  Adversarial: sigma/Delta ~ {adversarial_results["L_mean"]:.2f}')
print(f'  Required noise amplification: {ratio:.2f}x')
print(f'\nConclusion: Removing adversarial watermark requires {ratio:.2f}x more noise,')
print(f'which destroys image quality, breaking the paper\'s guarantee.')

## Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

methods = list(baseline_results.keys()) + ['Adversarial']
L_means = [baseline_results[k]['L_mean'] for k in baseline_results.keys()] + [adversarial_results['L_mean']]
L_stds = [baseline_results[k]['L_std'] for k in baseline_results.keys()] + [adversarial_results['L_std']]

# Plot 1: Lipschitz (top-left)
axes[0,0].bar(methods, L_means, yerr=L_stds, capsize=5)
axes[0,0].set_ylabel('Lipschitz Constant (L)')
axes[0,0].set_title('Lipschitz Constants')
axes[0,0].tick_params(axis='x', rotation=45)

pixel_means = [baseline_results[k]['pixel_dist_mean'] for k in baseline_results.keys()] + [adversarial_results['pixel_dist_mean']]
latent_means = [baseline_results[k]['latent_dist_mean'] for k in baseline_results.keys()] + [adversarial_results['latent_dist_mean']]

x = np.arange(len(methods))
width = 0.35
# Plot 2: Distances (top-right)
axes[0,1].bar(x - width/2, pixel_means, width, label='Pixel')
axes[0,1].bar(x + width/2, latent_means, width, label='Latent')
axes[0,1].set_ylabel('Distance')
axes[0,1].set_title('Pixel vs Latent Distance')
axes[0,1].set_xticks(x)
axes[0,1].set_xticklabels(methods, rotation=45)
axes[0,1].legend()

# Plot 3: PSNR (bottom-left)
psnr_means = [baseline_results[k]['psnr_mean'] for k in baseline_results.keys()] + [adversarial_results['psnr_mean']]
axes[1,0].bar(methods, psnr_means)
axes[1,0].set_ylabel('PSNR (dB)')
axes[1,0].set_title('PSNR (higher = better)')
axes[1,0].tick_params(axis='x', rotation=45)

# Plot 4: SSIM (bottom-right)
ssim_means = [baseline_results[k]['ssim_mean'] for k in baseline_results.keys()] + [adversarial_results['ssim_mean']]
axes[1,1].bar(methods, ssim_means)
axes[1,1].set_ylabel('SSIM')
axes[1,1].set_title('SSIM (higher = better)')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'lipschitz_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

## Save Results

In [ ]:
import json

results_summary = {
    'baseline': baseline_results,
    'adversarial': adversarial_results,
    'violation_ratio': ratio,
    'property_violated': bool(ratio > 5)
}

with open(os.path.join(output_dir, 'lipschitz_results.json'), 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f'\nResults saved to {output_dir}')